# 03. 코호트 리텐션 분석 (Cohort Retention Analysis)

**Dataset**: Google Merchandise Store (GA4 Public Dataset)  
**Period**: 2016-08-01 ~ 2017-08-01

## 목표
- 주간 코호트별 리텐션 매트릭스 구축
- 리텐션 히트맵 시각화
- 평균 리텐션 커브 산출
- 구매자 vs 비구매자 리텐션 비교
- Week 1 리텐션 드롭 분석

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.cloud import bigquery

sns.set_theme(style='whitegrid', palette='colorblind')
plt.rcParams['figure.dpi'] = 120

client = bigquery.Client()
print('BigQuery 연결 성공')

## 3.1 주간 코호트 리텐션 매트릭스

In [ ]:
query_retention = """
WITH first_visit AS (
  SELECT
    fullVisitorId,
    DATE_TRUNC(MIN(PARSE_DATE('%Y%m%d', date)), WEEK(MONDAY)) AS cohort_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId
),
user_activity AS (
  SELECT DISTINCT
    fullVisitorId,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', date), WEEK(MONDAY)) AS activity_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
),
retention_data AS (
  SELECT
    fv.cohort_week,
    DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) AS week_number,
    COUNT(DISTINCT fv.fullVisitorId) AS users
  FROM first_visit fv
  JOIN user_activity ua ON fv.fullVisitorId = ua.fullVisitorId
  WHERE DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) BETWEEN 0 AND 12
  GROUP BY cohort_week, week_number
),
cohort_size AS (
  SELECT cohort_week, users AS cohort_users
  FROM retention_data
  WHERE week_number = 0
)
SELECT
  rd.cohort_week,
  cs.cohort_users,
  rd.week_number,
  rd.users AS retained_users,
  ROUND(rd.users * 100.0 / cs.cohort_users, 2) AS retention_pct
FROM retention_data rd
JOIN cohort_size cs ON rd.cohort_week = cs.cohort_week
ORDER BY rd.cohort_week, rd.week_number
"""

df_ret = client.query(query_retention).to_dataframe()
df_ret['cohort_week'] = pd.to_datetime(df_ret['cohort_week'])
print(f"코호트 수: {df_ret['cohort_week'].nunique()}")
df_ret.head(15)

In [ ]:
# 리텐션 히트맵 생성
pivot = df_ret.pivot_table(
    index='cohort_week', columns='week_number',
    values='retention_pct', aggfunc='first'
)

# 최근 13개 코호트만 표시 (가독성)
pivot_display = pivot.tail(20)
pivot_display.index = pivot_display.index.strftime('%Y-%m-%d')

fig, ax = plt.subplots(figsize=(16, 10))
sns.heatmap(
    pivot_display,
    annot=True, fmt='.1f', cmap='YlOrRd_r',
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'Retention %'},
    ax=ax
)
ax.set_title('Weekly Cohort Retention Heatmap', fontsize=14)
ax.set_xlabel('Weeks Since First Visit')
ax.set_ylabel('Cohort (First Visit Week)')
plt.tight_layout()
plt.show()

## 3.2 평균 리텐션 커브

In [ ]:
query_avg_retention = """
WITH first_visit AS (
  SELECT
    fullVisitorId,
    DATE_TRUNC(MIN(PARSE_DATE('%Y%m%d', date)), WEEK(MONDAY)) AS cohort_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId
),
user_activity AS (
  SELECT DISTINCT
    fullVisitorId,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', date), WEEK(MONDAY)) AS activity_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
)
SELECT
  DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) AS week_number,
  COUNT(DISTINCT fv.fullVisitorId) AS retained_users
FROM first_visit fv
JOIN user_activity ua ON fv.fullVisitorId = ua.fullVisitorId
WHERE DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) BETWEEN 0 AND 12
GROUP BY week_number
ORDER BY week_number
"""

df_avg = client.query(query_avg_retention).to_dataframe()
df_avg['retention_pct'] = df_avg['retained_users'] / df_avg['retained_users'].iloc[0] * 100
df_avg

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df_avg['week_number'], df_avg['retention_pct'],
        marker='o', linewidth=2, color='#4e79a7', markersize=8)

# Week 1 drop 강조
week0 = df_avg['retention_pct'].iloc[0]
week1 = df_avg['retention_pct'].iloc[1]
ax.annotate(
    f'Week 1 Drop: {week0 - week1:.1f}pp',
    xy=(1, week1), xytext=(3, week1 + 15),
    arrowprops=dict(arrowstyle='->', color='#e15759'),
    fontsize=11, color='#e15759', fontweight='bold'
)

for i, row in df_avg.iterrows():
    ax.text(row['week_number'], row['retention_pct'] + 1.5,
            f"{row['retention_pct']:.1f}%", ha='center', fontsize=8)

ax.set_xlabel('Weeks Since First Visit')
ax.set_ylabel('Retention Rate (%)')
ax.set_title('Average Retention Curve', fontsize=14)
ax.set_ylim(0, 105)
ax.set_xticks(range(0, 13))
plt.tight_layout()
plt.show()

print(f"Week 0 → Week 1 리텐션 드롭: {week0 - week1:.1f}pp")
print(f"Week 1 리텐션: {week1:.1f}%")
print(f"Week 12 리텐션: {df_avg['retention_pct'].iloc[-1]:.1f}%")

## 3.3 구매자 vs 비구매자 리텐션

In [ ]:
query_buyer_ret = """
WITH purchasers AS (
  SELECT DISTINCT fullVisitorId
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`,
       UNNEST(hits) AS hits
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
    AND hits.eCommerceAction.action_type = '6'
),
first_visit AS (
  SELECT fullVisitorId,
    DATE_TRUNC(MIN(PARSE_DATE('%Y%m%d', date)), WEEK(MONDAY)) AS cohort_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
  GROUP BY fullVisitorId
),
user_activity AS (
  SELECT DISTINCT fullVisitorId,
    DATE_TRUNC(PARSE_DATE('%Y%m%d', date), WEEK(MONDAY)) AS activity_week
  FROM `bigquery-public-data.google_analytics_sample.ga_sessions_*`
  WHERE _TABLE_SUFFIX BETWEEN '20160801' AND '20170801'
),
classified AS (
  SELECT
    fv.fullVisitorId,
    DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) AS week_number,
    IF(p.fullVisitorId IS NOT NULL, 'purchaser', 'non_purchaser') AS user_type
  FROM first_visit fv
  JOIN user_activity ua ON fv.fullVisitorId = ua.fullVisitorId
  LEFT JOIN purchasers p ON fv.fullVisitorId = p.fullVisitorId
  WHERE DATE_DIFF(ua.activity_week, fv.cohort_week, WEEK) BETWEEN 0 AND 12
)
SELECT
  user_type,
  week_number,
  COUNT(DISTINCT fullVisitorId) AS retained_users
FROM classified
GROUP BY user_type, week_number
ORDER BY user_type, week_number
"""

df_buyer = client.query(query_buyer_ret).to_dataframe()

# 리텐션 % 계산
for ut in df_buyer['user_type'].unique():
    mask = df_buyer['user_type'] == ut
    base = df_buyer.loc[mask & (df_buyer['week_number'] == 0), 'retained_users'].values[0]
    df_buyer.loc[mask, 'retention_pct'] = df_buyer.loc[mask, 'retained_users'] / base * 100

df_buyer.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for ut, color, marker in [('purchaser', '#59a14f', 's'), ('non_purchaser', '#e15759', 'o')]:
    data = df_buyer[df_buyer['user_type'] == ut]
    label = 'Purchaser' if ut == 'purchaser' else 'Non-Purchaser'
    ax.plot(data['week_number'], data['retention_pct'],
            marker=marker, linewidth=2, label=label, color=color, markersize=7)

ax.set_xlabel('Weeks Since First Visit')
ax.set_ylabel('Retention Rate (%)')
ax.set_title('Retention: Purchaser vs Non-Purchaser', fontsize=14)
ax.set_xticks(range(0, 13))
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

# 리텐션 차이 정량화
purchaser_w1 = df_buyer[(df_buyer['user_type'] == 'purchaser') & (df_buyer['week_number'] == 1)]['retention_pct'].values[0]
non_purchaser_w1 = df_buyer[(df_buyer['user_type'] == 'non_purchaser') & (df_buyer['week_number'] == 1)]['retention_pct'].values[0]
print(f"\nWeek 1 리텐션 비교:")
print(f"  구매자: {purchaser_w1:.1f}%")
print(f"  비구매자: {non_purchaser_w1:.1f}%")
print(f"  차이: {purchaser_w1 - non_purchaser_w1:.1f}pp")
print(f"  구매자 리텐션이 {purchaser_w1 / non_purchaser_w1:.1f}x 높음")

## 3.4 코호트 크기 추이

In [ ]:
# 코호트별 신규 사용자 수 (pivot에서 추출)
cohort_sizes = df_ret[df_ret['week_number'] == 0][['cohort_week', 'cohort_users']].copy()

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(cohort_sizes['cohort_week'], cohort_sizes['cohort_users'], width=5, color='#4e79a7')
ax.set_xlabel('Cohort Week')
ax.set_ylabel('New Users')
ax.set_title('Weekly Cohort Sizes (New Users per Week)', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Key Findings

1. **Week 1 Drop**: 첫 방문 후 1주일 이내에 대다수의 사용자가 이탈
   - Week 0 → Week 1 리텐션이 가장 큰 하락 지점
   - 첫 방문 경험(onboarding)의 중요성을 보여줌

2. **구매자 리텐션 우위**: 구매 경험이 있는 사용자의 리텐션이 비구매자 대비 크게 높음
   - 첫 구매 유도가 장기 리텐션에 핵심적 영향
   - 첫 구매 할인 쿠폰, 무료 배송 등의 프로모션이 LTV 극대화에 효과적일 수 있음

3. **안정화 시점**: 일정 주차 이후 리텐션이 안정화되는 패턴
   - 초기 이탈을 막는 것이 핵심 → 3주 이내 재방문 유도 전략 필요

### Action Items
- 첫 방문 후 1주 이내 리타겟팅 이메일/푸시 전략
- 첫 구매 인센티브 프로그램 도입
- Week 2-3에 재방문 유도 캠페인